# Notebook 08: セキュアコーディングのまとめ

これまで学んだ脆弱性の知識を「守る側」の視点でまとめます。
安全なコードを書くための原則とチェックリストを身につけましょう。

## 学習目標
1. セキュアコーディングの基本原則を理解する
2. OWASP Top 10 の概要を知る
3. 実践的なセキュリティチェックリストを作る

---

**「攻撃を知ることは、防御の第一歩」**

## 1. セキュアコーディングの原則

### 原則 1: 入力を信頼しない
全てのユーザー入力は悪意がある可能性がある。

```python
# NG: 入力をそのまま使う
sql = f"SELECT * FROM users WHERE id = {request.args['id']}"

# OK: バリデーション + パラメータ化
user_id = int(request.args['id'])  # 数値以外はエラー
sql = "SELECT * FROM users WHERE id = ?"
db.execute(sql, (user_id,))
```

### 原則 2: 最小権限の原則
必要最小限の権限のみを与える。

```python
# NG: DBに全権限で接続
conn = connect(user='root', password='root_pw')

# OK: 読み取り専用ユーザーで接続
conn = connect(user='app_readonly', password='...')
```

### 原則 3: 多層防御
一つの防御が突破されても、他の防御で守る。

```
入力バリデーション → パラメータ化クエリ → WAF → エラーハンドリング
```

### 原則 4: 安全なデフォルト
デフォルトで安全な設定にする。

```python
# Jinja2 の autoescape=True がデフォルト → XSS 防止
# Django の CSRF ミドルウェアがデフォルト有効 → CSRF 防止
```

In [ ]:
# === 入力バリデーションの実装例 ===

import re

def validate_username(username: str) -> str:
    """ユーザー名のバリデーション"""
    if not username:
        raise ValueError("ユーザー名は必須です")
    if len(username) < 3 or len(username) > 20:
        raise ValueError("ユーザー名は3〜20文字にしてください")
    if not re.match(r'^[a-zA-Z0-9_]+$', username):
        raise ValueError("ユーザー名は英数字とアンダースコアのみ使用可能です")
    return username

def validate_email(email: str) -> str:
    """メールアドレスのバリデーション"""
    if not re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', email):
        raise ValueError("無効なメールアドレスです")
    return email

def validate_age(age_str: str) -> int:
    """年齢のバリデーション"""
    try:
        age = int(age_str)
    except ValueError:
        raise ValueError("年齢は数値で入力してください")
    if age < 0 or age > 150:
        raise ValueError("年齢は0〜150の範囲で入力してください")
    return age

# テスト
test_cases = [
    ("validate_username", validate_username, [
        ("alice", True),
        ("admin' --", False),
        ("<script>alert(1)</script>", False),
        ("ab", False),
        ("valid_user_123", True),
    ]),
    ("validate_email", validate_email, [
        ("alice@example.com", True),
        ("not_an_email", False),
        ("test@test", False),
    ]),
    ("validate_age", validate_age, [
        ("25", True),
        ("abc", False),
        ("-1", False),
        ("200", False),
    ]),
]

for func_name, func, cases in test_cases:
    print(f"=== {func_name} ===")
    for value, should_pass in cases:
        try:
            result = func(value)
            status = "PASS" if should_pass else "FAIL (should have rejected)"
            print(f"  '{value}' → {result} [{status}]")
        except ValueError as e:
            status = "PASS (rejected)" if not should_pass else "FAIL (should have accepted)"
            print(f"  '{value}' → Error: {e} [{status}]")
    print()

## 2. OWASP Top 10（2021）

OWASP（Open Web Application Security Project）は Web セキュリティの
最も重要な脅威をまとめたリストを公開しています。

| # | 脆弱性 | このコースでの対応 |
|---|--------|-------------------|
| A01 | アクセス制御の不備 | Notebook 05（認証） |
| A02 | 暗号化の失敗 | Notebook 05（パスワードハッシュ）、06（TLS） |
| A03 | インジェクション | Notebook 03（SQLi）、04（XSS） |
| A04 | 安全でない設計 | この Notebook |
| A05 | セキュリティ設定ミス | この Notebook |
| A06 | 脆弱で古いコンポーネント | 依存関係の管理 |
| A07 | 認証・識別の失敗 | Notebook 05 |
| A08 | ソフトウェア・データの整合性の不備 | サプライチェーン |
| A09 | セキュリティログ・監視の失敗 | ログの重要性 |
| A10 | SSRF（サーバーサイドリクエストフォージェリ） | サーバー側の入力検証 |

In [ ]:
# === セキュリティ設定ミスの例 ===

print("=== よくあるセキュリティ設定ミス ===")
print()

# 1. デバッグモードの本番有効化
print("1. デバッグモード")
print("   NG: app.run(debug=True)  # 本番環境")
print("   OK: app.run(debug=False) # 本番環境")
print("   → デバッグモードではスタックトレースやインタラクティブデバッガが露出")
print()

# 2. シークレットキーのハードコーディング
print("2. シークレットキー")
print('   NG: app.secret_key = "super_secret_123"')
print('   OK: app.secret_key = os.environ["SECRET_KEY"]')
print("   → コードに直書きするとGitHubで公開されてしまう")
print()

# 3. CORS の設定
print("3. CORS 設定")
print('   NG: Access-Control-Allow-Origin: *')
print('   OK: Access-Control-Allow-Origin: https://myapp.com')
print("   → ワイルドカードは全てのオリジンからのアクセスを許可")
print()

# 4. エラーメッセージ
print("4. エラーメッセージ")
print('   NG: return f"SQLエラー: {error}"  # 詳細なエラーを返す')
print('   OK: return "エラーが発生しました"  # ユーザーには一般的なメッセージ')
print("   → 詳細なエラーは攻撃者にシステムの情報を与える")

In [ ]:
# === 安全なパスワードポリシー ===

import re
import hashlib
import os

def check_password_strength(password: str) -> tuple[bool, list[str]]:
    """パスワードの強度をチェックする"""
    issues = []
    
    if len(password) < 8:
        issues.append("8文字以上にしてください")
    if not re.search(r'[A-Z]', password):
        issues.append("大文字を含めてください")
    if not re.search(r'[a-z]', password):
        issues.append("小文字を含めてください")
    if not re.search(r'[0-9]', password):
        issues.append("数字を含めてください")
    if not re.search(r'[!@#$%^&*(),.?":{}|<>]', password):
        issues.append("記号を含めてください")
    
    # よくあるパスワードのチェック
    common = ["password", "123456", "qwerty", "admin", "letmein"]
    if password.lower() in common:
        issues.append("よく使われるパスワードは避けてください")
    
    return len(issues) == 0, issues

# テスト
test_passwords = [
    "password",
    "abc123",
    "MyP@ssw0rd!",
    "Str0ng!P@ss#2024",
    "12345678",
]

print("=== パスワード強度チェック ===")
for pw in test_passwords:
    is_strong, issues = check_password_strength(pw)
    status = "STRONG" if is_strong else "WEAK"
    print(f"\n  '{pw}' → {status}")
    for issue in issues:
        print(f"    - {issue}")

## 3. セキュリティチェックリスト

### Web アプリ開発時のチェックリスト

#### 入力処理
- [ ] 全てのユーザー入力をバリデーションしているか
- [ ] SQL にはパラメータ化クエリを使用しているか
- [ ] HTML 出力はエスケープしているか（XSS 対策）
- [ ] ファイルアップロードの拡張子・サイズ制限はあるか

#### 認証・認可
- [ ] パスワードはハッシュ化して保存しているか（bcrypt/PBKDF2）
- [ ] セッション ID は暗号学的乱数で生成しているか
- [ ] Cookie に HttpOnly, Secure, SameSite を設定しているか
- [ ] 権限チェックは全てのエンドポイントで行っているか

#### 通信
- [ ] HTTPS を使用しているか
- [ ] HSTS ヘッダーを設定しているか
- [ ] CSP ヘッダーを設定しているか

#### 設定
- [ ] デバッグモードは本番で無効か
- [ ] シークレットキーは環境変数から読んでいるか
- [ ] エラーメッセージに内部情報が含まれていないか
- [ ] 不要なポートやサービスは閉じているか

#### 依存関係
- [ ] 使用しているライブラリに既知の脆弱性はないか
- [ ] 定期的に依存関係を更新しているか

In [ ]:
# === 脆弱なコード vs 安全なコード: 総まとめ ===

print("=" * 60)
print("セキュアコーディング総まとめ")
print("=" * 60)

examples = [
    {
        "topic": "SQLインジェクション（Notebook 03）",
        "bad":  'f"SELECT * FROM users WHERE id = {user_id}"',
        "good": '"SELECT * FROM users WHERE id = ?", (user_id,)',
    },
    {
        "topic": "XSS（Notebook 04）",
        "bad":  'f"<div>{user_input}</div>"',
        "good": 'f"<div>{html.escape(user_input)}</div>"',
    },
    {
        "topic": "パスワード保存（Notebook 05）",
        "bad":  'db.save(username, password)  # 平文',
        "good": 'db.save(username, bcrypt.hash(password))',
    },
    {
        "topic": "セッション（Notebook 05）",
        "bad":  'session_id = str(user.id)  # 推測可能',
        "good": 'session_id = secrets.token_hex(32)',
    },
    {
        "topic": "通信（Notebook 06）",
        "bad":  'http://example.com  # 暗号化なし',
        "good": 'https://example.com  # TLS暗号化',
    },
    {
        "topic": "エラーハンドリング",
        "bad":  'return f"Error: {traceback.format_exc()}"',
        "good": 'logger.error(e); return "エラーが発生しました"',
    },
]

for ex in examples:
    print(f"\n--- {ex['topic']} ---")
    print(f"  NG: {ex['bad']}")
    print(f"  OK: {ex['good']}")

## 4. 次のステップ

このコースで学んだことの振り返り:

| Notebook | 学んだこと | キーワード |
|----------|-----------|------------|
| 01 | セキュリティの基礎概念 | CIA三原則、脅威モデル |
| 02 | Webの仕組み | HTTP、Cookie、セッション |
| 03 | SQLインジェクション | パラメータ化クエリ |
| 04 | XSS | HTMLエスケープ、CSP |
| 05 | 認証の脆弱性 | bcrypt、JWT |
| 06 | ネットワークセキュリティ | TLS、DNS |
| 07 | CTF入門 | 暗号、フォレンジック |
| 08 | セキュアコーディング | OWASP、チェックリスト |

### 更に学ぶには
- **OWASP Web Security Testing Guide**: Web アプリのテスト方法
- **PortSwigger Web Security Academy**: 無料の Web セキュリティ学習
- **Hack The Box**: 実践的なペネトレーションテスト
- **CTF**: 実際の大会に参加して腕を磨く

## 最終演習

1. labs/vulnerable_app のコードを読んで、全ての脆弱性を見つけてリストアップしてみよう
2. 見つけた脆弱性に対する修正案を書いてみよう
3. 自分が過去に書いたコードにセキュリティチェックリストを適用してみよう
4. CTF に参加して、学んだ知識を実践してみよう